In [14]:
"""
PPO + LoRA fine-tuning loop using trl==0.11
GPU target: RTX 4070 16GB
"""

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType
from trl import PPOTrainer, PPOConfig, AutoModelForCausalLMWithValueHead, PPOv2Trainer, PPOv2Config
from trl.core import LengthSampler

In [15]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

Device: cuda


In [16]:
# ──────────────────────────────────────────────
# 1. MODEL REGISTRY  (5 quantised options)
# ──────────────────────────────────────────────
MODEL_OPTIONS = {
    "phi3-mini":      "microsoft/Phi-3-mini-4k-instruct",      # ~3.8B  – very fast
    "gemma2-2b":      "google/gemma-2-2b-it",                  # ~2.6B  – lightweight
    "gemma-3-270m": "google/gemma-3-270m-it",
    "llama3.2-3b":    "meta-llama/Llama-3.2-3B-Instruct",      # ~3.2B  – solid baseline
    "mistral-7b":     "mistralai/Mistral-7B-Instruct-v0.3",    # ~7B    – 4-bit needed
    "qwen2.5-7b":     "Qwen/Qwen2.5-7B-Instruct",             # ~7.6B  – strong instruct
}

# ──────────────────────────────────────────────
# ▶  SELECT YOUR MODEL HERE
# ──────────────────────────────────────────────
SELECTED_MODEL = "gemma-3-270m"


In [17]:
# ──────────────────────────────────────────────
# 2. QUANTISATION CONFIG
#    4-bit for 7B models, 8-bit for <4B models
# ──────────────────────────────────────────────
def get_bnb_config(model_key: str) -> BitsAndBytesConfig:
    large_models = {"mistral-7b", "qwen2.5-7b"}
    if model_key in large_models:
        return BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
        )
    else:
        return BitsAndBytesConfig(
            load_in_8bit=True,
            llm_int8_enable_fp32_cpu_offload=False,
        )

In [18]:
# 3. LORA CONFIG
# ──────────────────────────────────────────────
LORA_CONFIG = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    # Target the attention + MLP projections (works for most modern architectures)
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)


In [19]:
# ──────────────────────────────────────────────
# 4. PPO CONFIG
# ──────────────────────────────────────────────
PPO_CFG = PPOConfig(
    model_name=MODEL_OPTIONS[SELECTED_MODEL],
    learning_rate=1e-5,
    batch_size=8,               # number of (prompt, response) pairs per PPO step
    mini_batch_size=2,          # gradient-accumulation sub-batches
    gradient_accumulation_steps=4,
    ppo_epochs=4,               # inner optimisation epochs per batch
    max_grad_norm=0.5,
    kl_penalty="kl",            # or "full" for full KL
    init_kl_coef=0.2,
    adap_kl_ctrl=True,
    target_kl=6.0,
    seed=42,
    optimize_cuda_cache=True,
    log_with=None,              # swap to "wandb" or "tensorboard" if desired
)

# Number of environment samples to collect before each PPO step
ROLLOUT_BATCH_SIZE = PPO_CFG.batch_size  # keep in sync

/home/leonardo/github/synthetic-generation-alignment/.venv/lib/python3.10/site-packages/trl/trainer/ppo_config.py:207: FutureWarning: `PPOConfig` is deprecated and will be removed in the future. Please use `PPOv2Config` with `PPOv2Trainer` instead.
  warnings.warn(


In [20]:
# ──────────────────────────────────────────────
# 5. GENERATION KWARGS
# ──────────────────────────────────────────────
GENERATION_KWARGS = {
    "min_length": -1,
    "top_k": 0,
    "top_p": 0.95,
    "do_sample": True,
    "pad_token_id": None,       # filled in after tokenizer is loaded
    "max_new_tokens": 256,
}

In [21]:
# ──────────────────────────────────────────────
# 6. MODEL + TOKENIZER LOADING
# ──────────────────────────────────────────────
def load_model_and_tokenizer(model_key: str):
    model_id = MODEL_OPTIONS[model_key]
    bnb_config = get_bnb_config(model_key)

    print(f"[load] Loading tokenizer from {model_id}")
    tokenizer = AutoTokenizer.from_pretrained(model_id, padding_side="left")
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print(f"[load] Loading base model with quantisation …")
    base_model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )

    # Inject LoRA adapters
    base_model = get_peft_model(base_model, LORA_CONFIG)
    base_model.print_trainable_parameters()

    # Wrap with value head for PPO
    model = AutoModelForCausalLMWithValueHead.from_pretrained(base_model)

    return model, tokenizer

In [22]:
# ──────────────────────────────────────────────
# 7.  CHAT TEMPLATES  (one per model family)
# ──────────────────────────────────────────────
# Each template receives a list of message dicts:
#   [{"role": "system", "content": "…"}, {"role": "user", "content": "…"}]
# and returns the fully-formatted string that is fed to the tokeniser.
#
# The templates deliberately do NOT append the assistant turn-start token
# so the model generates it freely during the PPO rollout.
 
def _apply_llama3_template(messages: list[dict]) -> str:
    """
    Meta LLaMA-3 / LLaMA-3.2 instruct format.
    <|begin_of_text|>
    <|start_header_id|>system<|end_header_id|>\n\n{sys}<|eot_id|>
    <|start_header_id|>user<|end_header_id|>\n\n{user}<|eot_id|>
    <|start_header_id|>assistant<|end_header_id|>\n\n
    """
    out = "<|begin_of_text|>"
    for msg in messages:
        out += (
            f"<|start_header_id|>{msg['role']}<|end_header_id|>\n\n"
            f"{msg['content']}<|eot_id|>"
        )
    out += "<|start_header_id|>assistant<|end_header_id|>\n\n"
    return out
 
 
def _apply_mistral_template(messages: list[dict]) -> str:
    """
    Mistral v0.3 instruct format.
    [INST] {optional_sys + user} [/INST]
    Note: Mistral merges system into the first user turn.
    """
    system_text = ""
    turns = []
    for msg in messages:
        if msg["role"] == "system":
            system_text = msg["content"].strip() + "\n\n"
        elif msg["role"] == "user":
            turns.append(f"[INST] {system_text}{msg['content']} [/INST]")
            system_text = ""          # only prepend system to the first user turn
        elif msg["role"] == "assistant":
            turns.append(msg["content"])
    return " ".join(turns)
 
 
def _apply_phi3_template(messages: list[dict]) -> str:
    """
    Microsoft Phi-3 instruct format.
    <|system|>\n{sys}<|end|>\n<|user|>\n{user}<|end|>\n<|assistant|>\n
    """
    out = ""
    for msg in messages:
        role_tag = {"system": "<|system|>", "user": "<|user|>",
                    "assistant": "<|assistant|>"}[msg["role"]]
        out += f"{role_tag}\n{msg['content']}<|end|>\n"
    out += "<|assistant|>\n"
    return out
 
 
def _apply_gemma2_template(messages: list[dict]) -> str:
    """
    Google Gemma-2 instruct format.
    <bos><start_of_turn>user\n{optional_sys + user}<end_of_turn>\n
    <start_of_turn>model\n
    Note: Gemma-2 has no native system role; we prepend it to the user turn.
    """
    system_text = ""
    out = "<bos>"
    for msg in messages:
        if msg["role"] == "system":
            system_text = msg["content"].strip() + "\n\n"
        elif msg["role"] == "user":
            out += f"<start_of_turn>user\n{system_text}{msg['content']}<end_of_turn>\n"
            system_text = ""
        elif msg["role"] == "assistant":
            out += f"<start_of_turn>model\n{msg['content']}<end_of_turn>\n"
    out += "<start_of_turn>model\n"
    return out

def _apply_gemma3_template(messages: list[dict]) -> str:
    """
    Google Gemma-2 instruct format.
    <bos><start_of_turn>user\n{optional_sys + user}<end_of_turn>\n
    <start_of_turn>model\n
    Note: Gemma-2 has no native system role; we prepend it to the user turn.
    """
    system_text = ""
    out = "<bos>"
    for msg in messages:
        if msg["role"] == "system":
            system_text = msg["content"].strip() + "\n\n"
        elif msg["role"] == "user":
            out += f"<start_of_turn>user\n{system_text}{msg['content']}<end_of_turn>\n"
            system_text = ""
        elif msg["role"] == "assistant":
            out += f"<start_of_turn>model\n{msg['content']}<end_of_turn>\n"
    out += "<start_of_turn>model\n"
    return out
 
 
def _apply_qwen25_template(messages: list[dict]) -> str:
    """
    Qwen-2.5 instruct format (ChatML).
    <|im_start|>system\n{sys}<|im_end|>\n
    <|im_start|>user\n{user}<|im_end|>\n
    <|im_start|>assistant\n
    """
    out = ""
    for msg in messages:
        out += f"<|im_start|>{msg['role']}\n{msg['content']}<|im_end|>\n"
    out += "<|im_start|>assistant\n"
    return out
 
 
# Dispatch table: model key → template function
_TEMPLATE_FN = {
    "llama3.2-3b": _apply_llama3_template,
    "mistral-7b":  _apply_mistral_template,
    "phi3-mini":   _apply_phi3_template,
    "gemma2-2b":   _apply_gemma2_template,
    "gemma-3-270m": _apply_gemma3_template,
    "qwen2.5-7b":  _apply_qwen25_template,
}
 
 
def _build_raw_messages() -> list[dict]:
    """
    Return a list of message dicts, e.g.:
 
        return [
            {"role": "system",  "content": "You are a helpful assistant."},
            {"role": "user",    "content": "Explain gravity in one sentence."},
        ]
 
    For a single-turn prompt with no system message:
 
        return [{"role": "user", "content": prompt_string}]
 
    For multi-turn / few-shot:
 
        return [
            {"role": "system",    "content": "…"},
            {"role": "user",      "content": "…"},
            {"role": "assistant", "content": "…"},
            {"role": "user",      "content": "…"},   # ← model continues from here
        ]
    ──────────────────────────────────────────────────────────────
    """
    # return [
            # {"role": "system",  "content": "You are a system playing a guessing game. You are allowed to reply with only one token."},
            # {"role": "user",    "content": "cico was born in 1999 and we are currently in 2026. 2026-1999 = 27, so cico is 27 years old. you were born in 2011. how old are you? answer with only your age."},
        # ]
        
    return [
            {"role": "system",  "content": "You are a system playing a guessing game. You are allowed to reply with only one token."},
            {"role": "user",    "content": "cico was born in 1999 and we are currently in 2026. how old is cico?"},
            {"role": "assistant", "content": "27"},
            {"role": "user",      "content": "you were born in 2011 and we are currently in 2026. how old are you?"}
        ]
 
 
def get_prompt(model_key: str = SELECTED_MODEL) -> str:
    """
    Build a chat-formatted prompt string for the selected model.
 
    Calls _build_raw_messages() to get the role/content pairs, then
    applies the appropriate chat template for `model_key`.
 
    Parameters
    ----------
    model_key : str
        One of the keys in MODEL_OPTIONS / _TEMPLATE_FN.
 
    Returns
    -------
    str
        Fully formatted prompt, ready to be tokenised and fed to the model.
    """
    if model_key not in _TEMPLATE_FN:
        raise ValueError(
            f"No chat template registered for '{model_key}'. "
            f"Available: {list(_TEMPLATE_FN.keys())}"
        )
    messages = _build_raw_messages()
    return _TEMPLATE_FN[model_key](messages)

In [23]:
# ──────────────────────────────────────────────
# 7. Reward function
# ──────────────────────────────────────────────

def reward_function(prompt: str, response: str) -> float:
    """
    Compute a scalar reward given a (prompt, response) pair.
    Replace with your own intrinsic reward function.
    """
    try:
        correct_age = 15
        age_guessed = int(response.strip())
        reward =  - abs(correct_age - age_guessed)
    except:
        reward = -10
    return reward


In [24]:
# 8. ROLLOUT  →  collect N (prompt, response, reward) triples
# ──────────────────────────────────────────────
def collect_rollouts(ppo_trainer: PPOTrainer, tokenizer, n: int, model_key: str = SELECTED_MODEL):
    """
    Generate `n` responses and score them.

    Returns
    -------
    queries   : list[torch.Tensor]  – tokenised prompts
    responses : list[torch.Tensor]  – tokenised model outputs
    rewards   : list[torch.Tensor]  – scalar reward tensors
    """
    queries, responses, rewards = [], [], []

    GENERATION_KWARGS["pad_token_id"] = tokenizer.pad_token_id

    for _ in range(n):
        prompt_text = get_prompt(model_key)

        # Tokenise prompt
        input_ids = tokenizer.encode(prompt_text, return_tensors="pt").squeeze(0).to(device)

        # Generate response via PPOTrainer (handles ref-model tracking)
        response_ids = ppo_trainer.generate(
            input_ids, #.unsqueeze(0)
            **GENERATION_KWARGS,
        ).squeeze(0).to(device)

        # Decode only the newly generated tokens
        response_text = tokenizer.decode(
            response_ids[len(input_ids):], skip_special_tokens=True
        )

        print(f"Response: {response_text}")
        print("")
        print(f"Response length: {len(response_text)}")

        # Score
        r = reward_function(prompt_text, response_text)
        reward_tensor = torch.tensor(r, dtype=torch.float32).to(device)

        queries.append(input_ids)
        responses.append(response_ids[len(input_ids):])   # response tokens only
        rewards.append(reward_tensor)

    return queries, responses, rewards

In [25]:
# ──────────────────────────────────────────────
# 9. TRAINING LOOP
# ──────────────────────────────────────────────
def train(num_steps: int = 200):
    """
    Main PPO training loop.

    Parameters
    ----------
    num_steps : int
        Number of PPO update steps (each step consumes ROLLOUT_BATCH_SIZE samples).
    """
    model, tokenizer = load_model_and_tokenizer(SELECTED_MODEL)

    ppo_trainer = PPOTrainer(
        config=PPO_CFG,
        model=model,
        ref_model=None,   # trl auto-creates a frozen copy when None
        tokenizer=tokenizer,
    )

    print(f"\n{'='*60}")
    print(f"  Model   : {SELECTED_MODEL}  ({MODEL_OPTIONS[SELECTED_MODEL]})")
    print(f"  Steps   : {num_steps}")
    print(f"  Rollouts: {ROLLOUT_BATCH_SIZE} per step")
    print(f"{'='*60}\n")

    for step in range(num_steps):

        # ── 9a. Collect rollouts ──────────────────────────────────
        queries, responses, rewards = collect_rollouts(
            ppo_trainer, tokenizer, n=ROLLOUT_BATCH_SIZE, model_key=SELECTED_MODEL
        )

        # ── 9b. PPO update step ───────────────────────────────────
        stats = ppo_trainer.step(queries, responses, rewards)

        # ── 9c. Logging ───────────────────────────────────────────
        mean_reward = sum(r.item() for r in rewards) / len(rewards)
        print(
            f"[step {step+1:>4}/{num_steps}]  "
            f"mean_reward={mean_reward:+.4f}  "
            f"kl={stats.get('objective/kl', float('nan')):.4f}  "
            f"policy_loss={stats.get('ppo/loss/policy', float('nan')):.4f}"
        )

        ppo_trainer.log_stats(stats, {"query": queries, "response": responses}, rewards)

        # ── 9d. Periodic checkpoint ───────────────────────────────
        if (step + 1) % 50 == 0:
            ckpt_path = f"checkpoints/{SELECTED_MODEL}_step{step+1}"
            ppo_trainer.model.save_pretrained(ckpt_path)
            tokenizer.save_pretrained(ckpt_path)
            print(f"  ✔ Checkpoint saved → {ckpt_path}")

    # ── 10. Final save ────────────────────────────────────────────
    final_path = f"checkpoints/{SELECTED_MODEL}_final"
    ppo_trainer.model.save_pretrained(final_path)
    tokenizer.save_pretrained(final_path)
    print(f"\n✅ Training complete. Model saved to {final_path}")

In [26]:
train(num_steps=200)

[load] Loading tokenizer from google/gemma-3-270m-it
[load] Loading base model with quantisation …


Loading weights: 100%|██████████| 236/236 [00:00<00:00, 1060.14it/s]


trainable params: 3,796,992 || all params: 271,895,168 || trainable%: 1.3965

  Model   : gemma-3-270m  (google/gemma-3-270m-it)
  Steps   : 200
  Rollouts: 8 per step



/home/leonardo/github/synthetic-generation-alignment/.venv/lib/python3.10/site-packages/trl/trainer/ppo_trainer.py:193: FutureWarning: `PPOTrainer` is deprecated and will be removed in trl v0.12. Please use `PPOv2Trainer` instead.
  warnings.warn(
/home/leonardo/github/synthetic-generation-alignment/.venv/lib/python3.10/site-packages/trl/trainer/ppo_trainer.py:273: UserWarning: No dataset is provided. Make sure to set config.batch_size to the correct value before training.
  warnings.warn(
/home/leonardo/github/synthetic-generation-alignment/.venv/lib/python3.10/site-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Response: I am currently 2026.


Response length: 21
Response: 27

Response length: 2
Response: I am currently in 2026.

Response length: 23
Response: I am currently 27 years old.

Response length: 28
Response: You are currently 27.


Response length: 22
Response: You are currently 27 years old.

Response length: 31
Response: You are currently 27 years old.


Response length: 32
Response: You are currently 27 years old.


Response length: 32
[step    1/200]  mean_reward=-10.2500  kl=0.0000  policy_loss=0.0847
Response: 27


Response length: 3
Response: 27

Response length: 2
Response: 27

Response length: 2
Response: 27


Response length: 3
Response: 27


Response length: 3
Response: 27

Response length: 2
Response: 27


Response length: 3
Response: 27


Response length: 3
[step    2/200]  mean_reward=-12.0000  kl=-0.3042  policy_loss=-0.0020
Response: I am currently 27 years old.


Response length: 29
Response: You are currently 27 years old.


Response length: 32
Response: 27


Respo

KeyboardInterrupt: 